In [1]:
import polars as pl
from polars import col
from investment_strategy.data.cleaner import fill_OHLCV_missing_values
from investment_strategy.signals.signal_construction import *
from investment_strategy.signals.signal_ranking import *
from investment_strategy.portfolio.weighting import *
from investment_strategy.portfolio.rebalancing import *
from investment_strategy.portfolio.valuation import *
from investment_strategy.analytics.return_metrics import *
from investment_strategy.analytics.risk_metrics import *
from datetime import date

market_data = pl.read_parquet("../data/raw/sp500_market_data.parquet")
market_data

date,ticker,open,high,low,close,volume
date,str,f64,f64,f64,f64,i64
2018-12-31,"""A""",62.795802,63.874904,62.795802,63.855968,1572100
2018-12-31,"""AAPL""",37.613949,37.810881,37.127551,37.42651,140014000
2018-12-31,"""ABBV""",66.040205,67.042343,65.773452,66.465576,5722100
2018-12-31,"""ABNB""",null,null,null,null,null
2018-12-31,"""ABT""",62.168729,63.202416,62.090555,62.828899,6094300
…,…,…,…,…,…,…
2026-05-29,"""XYZ""",74.970001,76.660004,74.195,75.720001,7380400
2026-05-29,"""YUM""",149.229996,150.179993,147.339996,147.949997,3992700
2026-05-29,"""ZBH""",81.952229,82.979498,81.363796,82.111809,3216600


# Signal Construction

In [2]:
market_data = fill_OHLCV_missing_values(market_data)
close_price = market_data.select(
    col("date"),
    col("ticker"),
    col("close")
)
close_price

date,ticker,close
date,str,f64
2018-12-31,"""A""",63.855968
2018-12-31,"""AAPL""",37.42651
2018-12-31,"""ABBV""",66.465576
2018-12-31,"""ABT""",62.828899
2018-12-31,"""ACGL""",25.408009
…,…,…
2026-05-29,"""XYZ""",75.720001
2026-05-29,"""YUM""",147.949997
2026-05-29,"""ZBH""",82.111809


In [3]:
rebalance_frequency = 2
backtest_period = 36
past_n_months = 6
unit = "mo"
start_date = date(2021, 12, 31)
rolling_window = 126

In [4]:
end_date = get_backtest_end_date(start_date, backtest_period, unit)
end_date

datetime.date(2024, 12, 31)

In [5]:
trading_calendar = get_trading_calendar(close_price)
trading_calendar

date
date
2018-12-31
2019-01-02
2019-01-03
2019-01-04
2019-01-07
…
2026-05-22
2026-05-26
2026-05-27


In [6]:
date_mapping_df = create_date_mapping(trading_calendar, rebalance_frequency, unit, start_date, end_date, past_n_months, unit)
date_mapping_df

lookback_date,signal_date,rebalance_date
date,date,date
2021-06-30,2021-12-30,2021-12-31
2021-08-25,2022-02-25,2022-02-28
2021-10-29,2022-04-29,2022-05-02
2021-12-29,2022-06-29,2022-06-30
2022-02-28,2022-08-30,2022-08-31
…,…,…
2023-10-27,2024-04-29,2024-04-30
2023-12-28,2024-06-28,2024-07-01
2024-02-29,2024-08-30,2024-09-03


In [7]:
price_date_df = get_prices_for_date_mapping(
    close_price, market_data, date_mapping_df
)
price_date_df

signal_date,ticker,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open
date,str,f64,date,date,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-12-31,142.468719,154.951484
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-12-31,133.502045,174.107375
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-12-31,92.721123,114.60431
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-12-31,153.139999,168.779999
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-12-31,104.788086,128.427822
…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-12-31,64.489998,87.720001
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-12-31,127.461578,130.30221
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-12-31,106.416763,104.312847


In [8]:
past_returns = calculate_past_returns(price_date_df, past_n_months, unit)
past_returns

signal_date,ticker,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns
date,str,f64,date,date,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-12-31,142.468719,154.951484,0.091212
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-12-31,133.502045,174.107375,0.304961
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-12-31,92.721123,114.60431,0.235012
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-12-31,153.139999,168.779999,0.102129
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-12-31,104.788086,128.427822,0.225595
…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-12-31,64.489998,87.720001,0.356489
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-12-31,127.461578,130.30221,0.017941
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-12-31,106.416763,104.312847,-0.024491


In [9]:
past_std = calculate_past_returns_std(
    close_price, trading_calendar, date_mapping_df, rolling_window, end_date
)
past_std

date,ticker,close,last_day_close,daily_return,past 126 trading days std
date,str,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,155.241348,0.001432,0.203429
2021-12-30,"""AAPL""",174.214935,175.36853,-0.006578,0.225141
2021-12-30,"""ABBV""",114.511681,114.031494,0.004211,0.195823
2021-12-30,"""ABNB""",168.779999,167.440002,0.008003,0.466457
2021-12-30,"""ABT""",128.427795,128.600876,-0.001346,0.170945
…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,88.970001,-0.016747,0.451158
2024-12-30,"""YUM""",129.748352,131.410034,-0.012645,0.165812
2024-12-30,"""ZBH""",103.810539,105.021988,-0.011535,0.234709


In [10]:
full_date_price_std_table = get_full_date_price_std_table(past_returns, past_std)
full_date_price_std_table

signal_date,ticker,past 126 trading days std,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns
date,str,f64,f64,date,date,f64,f64,f64
2021-12-30,"""A""",0.203429,155.463638,2021-06-30,2021-12-31,142.468719,154.951484,0.091212
2021-12-30,"""AAPL""",0.225141,174.214935,2021-06-30,2021-12-31,133.502045,174.107375,0.304961
2021-12-30,"""ABBV""",0.195823,114.511681,2021-06-30,2021-12-31,92.721123,114.60431,0.235012
2021-12-30,"""ABNB""",0.466457,168.779999,2021-06-30,2021-12-31,153.139999,168.779999,0.102129
2021-12-30,"""ABT""",0.170945,128.427795,2021-06-30,2021-12-31,104.788086,128.427822,0.225595
…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",0.451158,87.480003,2024-06-28,2024-12-31,64.489998,87.720001,0.356489
2024-12-30,"""YUM""",0.165812,129.748352,2024-06-28,2024-12-31,127.461578,130.30221,0.017941
2024-12-30,"""ZBH""",0.234709,103.810539,2024-06-28,2024-12-31,106.416763,104.312847,-0.024491


In [11]:
risk_adjusted_table = get_risk_adjusted_return(full_date_price_std_table, past_n_months, unit, rolling_window)
risk_adjusted_table

signal_date,ticker,past 126 trading days std,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns,risk_adjusted past 6mo total return
date,str,f64,f64,date,date,f64,f64,f64,f64
2021-12-30,"""A""",0.203429,155.463638,2021-06-30,2021-12-31,142.468719,154.951484,0.091212,0.448375
2021-12-30,"""AAPL""",0.225141,174.214935,2021-06-30,2021-12-31,133.502045,174.107375,0.304961,1.35453
2021-12-30,"""ABBV""",0.195823,114.511681,2021-06-30,2021-12-31,92.721123,114.60431,0.235012,1.200126
2021-12-30,"""ABNB""",0.466457,168.779999,2021-06-30,2021-12-31,153.139999,168.779999,0.102129,0.218946
2021-12-30,"""ABT""",0.170945,128.427795,2021-06-30,2021-12-31,104.788086,128.427822,0.225595,1.319692
…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",0.451158,87.480003,2024-06-28,2024-12-31,64.489998,87.720001,0.356489,0.790165
2024-12-30,"""YUM""",0.165812,129.748352,2024-06-28,2024-12-31,127.461578,130.30221,0.017941,0.1082
2024-12-30,"""ZBH""",0.234709,103.810539,2024-06-28,2024-12-31,106.416763,104.312847,-0.024491,-0.104345


# Signal Ranking

In [12]:
risk_adjusted_table.with_columns(
    col("risk_adjusted past 6mo total return").rank(method="ordinal").over("signal_date").alias("signal_rank")
)

signal_date,ticker,past 126 trading days std,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns,risk_adjusted past 6mo total return,signal_rank
date,str,f64,f64,date,date,f64,f64,f64,f64,u32
2021-12-30,"""A""",0.203429,155.463638,2021-06-30,2021-12-31,142.468719,154.951484,0.091212,0.448375,255
2021-12-30,"""AAPL""",0.225141,174.214935,2021-06-30,2021-12-31,133.502045,174.107375,0.304961,1.35453,454
2021-12-30,"""ABBV""",0.195823,114.511681,2021-06-30,2021-12-31,92.721123,114.60431,0.235012,1.200126,430
2021-12-30,"""ABNB""",0.466457,168.779999,2021-06-30,2021-12-31,153.139999,168.779999,0.102129,0.218946,188
2021-12-30,"""ABT""",0.170945,128.427795,2021-06-30,2021-12-31,104.788086,128.427822,0.225595,1.319692,450
…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",0.451158,87.480003,2024-06-28,2024-12-31,64.489998,87.720001,0.356489,0.790165,369
2024-12-30,"""YUM""",0.165812,129.748352,2024-06-28,2024-12-31,127.461578,130.30221,0.017941,0.1082,181
2024-12-30,"""ZBH""",0.234709,103.810539,2024-06-28,2024-12-31,106.416763,104.312847,-0.024491,-0.104345,139


In [13]:
signal_ranked = rank_signal(risk_adjusted_table, f"risk_adjusted past {past_n_months}{unit} total return")
signal_ranked

signal_date,ticker,past 126 trading days std,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns,risk_adjusted past 6mo total return,risk_adjusted past 6mo total return rank
date,str,f64,f64,date,date,f64,f64,f64,f64,u32
2021-12-30,"""A""",0.203429,155.463638,2021-06-30,2021-12-31,142.468719,154.951484,0.091212,0.448375,238
2021-12-30,"""AAPL""",0.225141,174.214935,2021-06-30,2021-12-31,133.502045,174.107375,0.304961,1.35453,39
2021-12-30,"""ABBV""",0.195823,114.511681,2021-06-30,2021-12-31,92.721123,114.60431,0.235012,1.200126,63
2021-12-30,"""ABNB""",0.466457,168.779999,2021-06-30,2021-12-31,153.139999,168.779999,0.102129,0.218946,305
2021-12-30,"""ABT""",0.170945,128.427795,2021-06-30,2021-12-31,104.788086,128.427822,0.225595,1.319692,43
…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",0.451158,87.480003,2024-06-28,2024-12-31,64.489998,87.720001,0.356489,0.790165,131
2024-12-30,"""YUM""",0.165812,129.748352,2024-06-28,2024-12-31,127.461578,130.30221,0.017941,0.1082,319
2024-12-30,"""ZBH""",0.234709,103.810539,2024-06-28,2024-12-31,106.416763,104.312847,-0.024491,-0.104345,361


In [14]:
filtered_ticker = filter_top_ranked(signal_ranked, f"risk_adjusted past {past_n_months}{unit} total return rank", 10)
filtered_ticker

signal_date,ticker,past 126 trading days std,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns,risk_adjusted past 6mo total return,risk_adjusted past 6mo total return rank
date,str,f64,f64,date,date,f64,f64,f64,f64,u32
2021-12-30,"""ACN""",0.205886,382.076599,2021-06-30,2021-12-31,270.601135,381.033429,0.411955,2.000885,6
2021-12-30,"""BLDR""",0.333324,84.050003,2021-06-30,2021-12-31,42.66,84.290001,0.97023,2.910771,1
2021-12-30,"""COST""",0.199371,535.266479,2021-06-30,2021-12-31,374.264069,534.791945,0.430184,2.157708,4
2021-12-30,"""CPT""",0.180843,151.312561,2021-06-30,2021-12-31,111.497383,151.304059,0.357095,1.974614,7
2021-12-30,"""EXC""",0.155709,34.810818,2021-06-30,2021-12-31,26.495146,34.54374,0.313856,2.015662,5
…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""GEV""",0.467186,329.117523,2024-06-28,2024-12-31,170.788254,329.785165,0.92705,1.984328,6
2024-12-30,"""KMI""",0.212926,25.715519,2024-06-28,2024-12-31,18.214417,25.771911,0.411822,1.93411,9
2024-12-30,"""NI""",0.148419,35.226597,2024-06-28,2024-12-31,27.213289,35.274594,0.294463,1.983997,7


In [15]:
sorted_ranking = sort_rankings(filtered_ticker, f"risk_adjusted past {past_n_months}{unit} total return rank")
sorted_ranking

signal_date,ticker,past 126 trading days std,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns,risk_adjusted past 6mo total return,risk_adjusted past 6mo total return rank
date,str,f64,f64,date,date,f64,f64,f64,f64,u32
2021-12-30,"""BLDR""",0.333324,84.050003,2021-06-30,2021-12-31,42.66,84.290001,0.97023,2.910771,1
2021-12-30,"""FDS""",0.169713,461.847412,2021-06-30,2021-12-31,318.493347,461.847424,0.450101,2.652124,2
2021-12-30,"""PLD""",0.172817,146.698746,2021-06-30,2021-12-31,103.488182,146.794842,0.417541,2.416083,3
2021-12-30,"""COST""",0.199371,535.266479,2021-06-30,2021-12-31,374.264069,534.791945,0.430184,2.157708,4
2021-12-30,"""EXC""",0.155709,34.810818,2021-06-30,2021-12-31,26.495146,34.54374,0.313856,2.015662,5
…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""GEV""",0.467186,329.117523,2024-06-28,2024-12-31,170.788254,329.785165,0.92705,1.984328,6
2024-12-30,"""NI""",0.148419,35.226597,2024-06-28,2024-12-31,27.213289,35.274594,0.294463,1.983997,7
2024-12-30,"""FISV""",0.197021,206.270004,2024-06-28,2024-12-31,149.039993,206.770004,0.383991,1.94899,8


# Portfolio Weights

In [16]:
initial_capital = 1_000_000

In [17]:
equal_weighted_portfolio = construct_portfolio_weights(filtered_ticker, "equal_weighted")
equal_weighted_portfolio

signal_date,ticker,past 126 trading days std,signal_close,lookback_date,rebalance_date,lookback_close,rebalance_open,past 6mo total returns,risk_adjusted past 6mo total return,risk_adjusted past 6mo total return rank,portfolio_weight
date,str,f64,f64,date,date,f64,f64,f64,f64,u32,f64
2021-12-30,"""ACN""",0.205886,382.076599,2021-06-30,2021-12-31,270.601135,381.033429,0.411955,2.000885,6,0.1
2021-12-30,"""BLDR""",0.333324,84.050003,2021-06-30,2021-12-31,42.66,84.290001,0.97023,2.910771,1,0.1
2021-12-30,"""COST""",0.199371,535.266479,2021-06-30,2021-12-31,374.264069,534.791945,0.430184,2.157708,4,0.1
2021-12-30,"""CPT""",0.180843,151.312561,2021-06-30,2021-12-31,111.497383,151.304059,0.357095,1.974614,7,0.1
2021-12-30,"""EXC""",0.155709,34.810818,2021-06-30,2021-12-31,26.495146,34.54374,0.313856,2.015662,5,0.1
…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""GEV""",0.467186,329.117523,2024-06-28,2024-12-31,170.788254,329.785165,0.92705,1.984328,6,0.1
2024-12-30,"""KMI""",0.212926,25.715519,2024-06-28,2024-12-31,18.214417,25.771911,0.411822,1.93411,9,0.1
2024-12-30,"""NI""",0.148419,35.226597,2024-06-28,2024-12-31,27.213289,35.274594,0.294463,1.983997,7,0.1


# Portfolio Construction

In [18]:
rebalance_allocation_df = prepare_rebalance_allocation_df(equal_weighted_portfolio)
rebalance_allocation_df

rebalance_date,ticker,rebalance_open,portfolio_weight
date,str,f64,f64
2021-12-31,"""ACN""",381.033429,0.1
2021-12-31,"""BLDR""",84.290001,0.1
2021-12-31,"""COST""",534.791945,0.1
2021-12-31,"""CPT""",151.304059,0.1
2021-12-31,"""EXC""",34.54374,0.1
…,…,…,…
2024-12-31,"""GEV""",329.785165,0.1
2024-12-31,"""KMI""",25.771911,0.1
2024-12-31,"""NI""",35.274594,0.1


In [19]:
rebalance_date = date_mapping_df.get_column("rebalance_date")
rebalance_date


rebalance_date
date
2021-12-31
2022-02-28
2022-05-02
2022-06-30
2022-08-31
…
2024-04-30
2024-07-01
2024-09-03


In [20]:
rebalance_summary = run_rebalance_simulation(
    price_date_df, rebalance_allocation_df, initial_capital, rebalance_date
)
rebalance_summary

{'rebalance_level_table': shape: (19, 3)
 ┌────────────────┬─────────────────┬───────────────┐
 │ rebalance_date ┆ portfolio_value ┆ cash_residual │
 │ ---            ┆ ---             ┆ ---           │
 │ date           ┆ f64             ┆ f64           │
 ╞════════════════╪═════════════════╪═══════════════╡
 │ 2021-12-31     ┆ 1e6             ┆ 1306.363865   │
 │ 2022-02-28     ┆ 881750.476647   ┆ 516.090575    │
 │ 2022-05-02     ┆ 969050.177535   ┆ 718.051758    │
 │ 2022-06-30     ┆ 906752.856855   ┆ 381.73067     │
 │ 2022-08-31     ┆ 986226.29668    ┆ 569.012048    │
 │ …              ┆ …               ┆ …             │
 │ 2024-04-30     ┆ 1.8918e6        ┆ 1142.972396   │
 │ 2024-07-01     ┆ 1.9268e6        ┆ 573.037835    │
 │ 2024-09-03     ┆ 1.9992e6        ┆ 450.164365    │
 │ 2024-10-31     ┆ 2.0734e6        ┆ 2049.644278   │
 │ 2024-12-31     ┆ 2.2487e6        ┆ 919.606129    │
 └────────────────┴─────────────────┴───────────────┘,
 'position_level_table': shape: (190, 3)

# Portfolio Valuation

In [21]:
rebalance_period_close_prices = get_backtest_period_close_prices(close_price, start_date, end_date)
rebalance_period_close_prices

date,ticker,close
date,str,f64
2021-12-31,"""A""",154.27504
2021-12-31,"""AAPL""",173.599014
2021-12-31,"""ABBV""",114.065155
2021-12-31,"""ABNB""",166.490005
2021-12-31,"""ABT""",128.19101
…,…,…
2024-12-31,"""XYZ""",84.989998
2024-12-31,"""YUM""",130.370239
2024-12-31,"""ZBH""",104.037064


In [22]:
next_date_matched_rebalance_level_table = get_next_date_matched_rebalance_level_table(rebalance_summary["rebalance_level_table"])
next_date_matched_rebalance_level_table

rebalance_date,portfolio_value,cash_residual,next_rebalance_date
date,f64,f64,date
2021-12-31,1e6,1306.363865,2022-02-28
2022-02-28,881750.476647,516.090575,2022-05-02
2022-05-02,969050.177535,718.051758,2022-06-30
2022-06-30,906752.856855,381.73067,2022-08-31
2022-08-31,986226.29668,569.012048,2022-10-31
…,…,…,…
2024-04-30,1.8918e6,1142.972396,2024-07-01
2024-07-01,1.9268e6,573.037835,2024-09-03
2024-09-03,1.9992e6,450.164365,2024-10-31


In [23]:
daily_position_value_table = get_daily_position_value_table(rebalance_period_close_prices, next_date_matched_rebalance_level_table, rebalance_summary["position_level_table"])
daily_position_value_table

date,rebalance_date,ticker,shares,close,position_value
date,date,str,i64,f64,f64
2021-12-31,2021-12-31,"""ACN""",262,382.741455,100278.26123
2021-12-31,2021-12-31,"""BLDR""",1186,85.709999,101652.058914
2021-12-31,2021-12-31,"""COST""",186,538.864075,100228.717896
2021-12-31,2021-12-31,"""CPT""",660,151.737152,100146.520386
2021-12-31,2021-12-31,"""EXC""",2894,35.059681,101462.716637
…,…,…,…,…,…
2024-12-31,2024-12-31,"""GEV""",681,327.792084,223226.409027
2024-12-31,2024-12-31,"""KMI""",8725,25.753113,224695.909119
2024-12-31,2024-12-31,"""NI""",6374,35.284191,224901.434273


In [24]:
daily_portfolio_table = get_daily_portfolio_table(next_date_matched_rebalance_level_table, daily_position_value_table)
daily_portfolio_table

date,positions_value,cash_residual,portfolio_value,daily_return
date,f64,f64,f64,f64
2021-12-31,1.0051e6,1306.363865,1.0064e6,null
2022-01-03,985838.075142,1306.363865,987144.439007,-0.019175
2022-01-04,987767.750992,1306.363865,989074.114857,0.001955
2022-01-05,964492.488579,1306.363865,965798.852444,-0.023532
2022-01-06,956307.677521,1306.363865,957614.041386,-0.008475
…,…,…,…,…
2024-12-24,2.2907e6,2049.644278,2.2928e6,0.005603
2024-12-26,2.2920e6,2049.644278,2.2940e6,0.000551
2024-12-27,2.2611e6,2049.644278,2.2632e6,-0.01344


# Analytics

In [25]:
daily_portfolio_price_return_df = prepare_daily_portfolio_price_return_df(daily_portfolio_table)
daily_portfolio_price_return_df

date,portfolio_value,daily_return
date,f64,f64
2021-12-31,1.0064e6,null
2022-01-03,987144.439007,-0.019175
2022-01-04,989074.114857,0.001955
2022-01-05,965798.852444,-0.023532
2022-01-06,957614.041386,-0.008475
…,…,…
2024-12-24,2.2928e6,0.005603
2024-12-26,2.2940e6,0.000551
2024-12-27,2.2632e6,-0.01344


In [26]:
total_return = calculate_total_return(daily_portfolio_price_return_df)
total_return

1.211639556701725

In [27]:
annualized_return_CAGR = calculate_annualized_return_CAGR(daily_portfolio_price_return_df)
annualized_return_CAGR

0.3042551821900754

In [28]:
mean_daily_return = calculate_mean_daily_return(daily_portfolio_price_return_df)
mean_daily_return

0.0011763516165092457

In [29]:
annualized_mean_return = calculate_annualized_mean_return(daily_portfolio_price_return_df)
annualized_mean_return

0.3448283509316117

In [30]:
mean_daily_std = calculate_daily_return_std(daily_portfolio_price_return_df)
mean_daily_std

0.015608500225834139

In [31]:
annualized_volatility = calculate_annualized_volatility(daily_portfolio_price_return_df)
annualized_volatility

0.2477772596175158

In [32]:
drawdown = calculate_drawdown(daily_portfolio_price_return_df)
drawdown

date,portfolio_value,daily_return,drawdown
date,f64,f64,f64
2021-12-31,1.0064e6,null,0.0
2022-01-03,987144.439007,-0.019175,-0.019175
2022-01-04,989074.114857,0.001955,-0.017257
2022-01-05,965798.852444,-0.023532,-0.040384
2022-01-06,957614.041386,-0.008475,-0.048516
…,…,…,…
2024-12-24,2.2928e6,0.005603,-0.075439
2024-12-26,2.2940e6,0.000551,-0.074929
2024-12-27,2.2632e6,-0.01344,-0.087362


In [33]:
max_drawdown = calculate_max_drawdown(drawdown)
max_drawdown

-0.1685877781521068

In [34]:
sharpe_ratio = calculate_sharpe_ratio(mean_daily_return, annualized_volatility)
sharpe_ratio

1.1963995720104978